# 04 Mutual Information Extension

## Aim
Explore a nonlinear dependence construction inspired by Donges et al. using a simple binned mutual information estimator.

## Method
The notebook loads the preprocessed Pearson matrix output, samples a smaller subset of nodes, computes pairwise binned mutual information, and constructs another fixed-density undirected graph.

## Outputs
- `data/processed/mutual_information_subset.npy`
- `data/processed/mutual_information_subset_adjacency.npy`
- `data/processed/mutual_information_subset_network.graphml`

## Brief Interpretation
The mutual information network is an optional prototype extension. Because the estimator is quadratic in the number of nodes, this notebook deliberately runs on a subset rather than the full grid.


In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

%reload_ext autoreload
%autoreload 2

import networkx as nx
import numpy as np
import pandas as pd

from src.config import EDGE_DENSITY, INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from src.config import RANDOM_SEED
from src.dependence import compute_mutual_information_matrix
from src.network_construction import adjacency_from_fixed_density
from src.network_construction import build_graph_from_adjacency

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


## Select a Computationally Manageable Node Subset

The subset is random but reproducible. Increase `max_nodes` only after confirming runtime is acceptable.


In [5]:
X = np.load(INTERIM_DATA_DIR / "X_preprocessed.npy")
node_metadata = pd.read_parquet(INTERIM_DATA_DIR / "node_metadata.parquet")

max_nodes = 200
n_subset = min(max_nodes, X.shape[1])
rng = np.random.default_rng(RANDOM_SEED)
subset = np.sort(rng.choice(X.shape[1], size=n_subset, replace=False))

X_subset = X[:, subset]
metadata_subset = node_metadata.iloc[subset].copy().reset_index(drop=True)
metadata_subset["original_filtered_node_id"] = metadata_subset["node_id"].to_numpy()
metadata_subset["node_id"] = np.arange(n_subset)

print(f"MI subset matrix shape: {X_subset.shape}")


MI subset matrix shape: (612, 200)


## Build the Mutual Information Network

Mutual information scores are non-negative, so the fixed-density adjacency is built from the largest MI values directly.


In [6]:
mi = compute_mutual_information_matrix(X_subset, n_bins=16)
np.save(PROCESSED_DATA_DIR / "mutual_information_subset.npy", mi)

A_mi = adjacency_from_fixed_density(mi, density=EDGE_DENSITY, use_absolute=False)
np.save(PROCESSED_DATA_DIR / "mutual_information_subset_adjacency.npy", A_mi)

G_mi = build_graph_from_adjacency(A_mi, metadata_subset)
nx.write_graphml(
    G_mi,
    PROCESSED_DATA_DIR / "mutual_information_subset_network.graphml",
)

print(G_mi)
print(f"Actual edge density: {nx.density(G_mi):.4f}")


Mutual information rows: 100%|██████████| 200/200 [00:06<00:00, 29.59it/s] 

Graph with 200 nodes and 199 edges
Actual edge density: 0.0100


## Interpretation

The MI graph should be treated as exploratory at this stage. A more careful nonlinear analysis would test estimator sensitivity to bin count, sample size, seasonal structure, and edge density.
